# BM25 using CSC Matrix
- Compressed Sparse Column (CSC) formatted Matrix
  - column-based compression for the almost zero-filled matrix

In [1]:
import os
import numpy as np

import marisa_trie
import pydantic
from scipy.sparse import csc_matrix

## CSC Matrix Examples

In [2]:
from scipy.sparse import csc_matrix
import numpy as np

# (1) csc matrix from dense array
dense = np.array([[0, 0, 1],
                  [4, 0, 0],
                  [0, 5, 6]])
A = csc_matrix(dense)

# (2) csc matrix from (data, (row, col)) format
data = [1, 4, 5, 6]  # non-zero entries
rows = [0, 1, 2, 2]  # row indices of non-zero entries
cols = [2, 0, 1, 2]  # column indices of non-zero entries
B = csc_matrix((data, (rows, cols)), shape=(3, 3))

print(A.toarray())
print(B.toarray())


[[0 0 1]
 [4 0 0]
 [0 5 6]]
[[0 0 1]
 [4 0 0]
 [0 5 6]]


In [3]:
(
    A.data,  # non-zero entries
    A.indices,  # row indices of non-zero entries
    A.indptr  # column pointer (index in data/indices where each column starts),
                # indptr has length n_cols + 1
                # e.g., for 3 columns, indptr has length 4
                # indptr[0] = 0 (start of col 0)
                # indptr[1] = 1 (start of col 1)
                # indptr[2] = 2 (start of col 2)
                # indptr[3] = 4 (end of col 2, total number of non-zero entries)
                # so col 0 has 1 entry, col 1 has 1 entry, col 2 has 2 entries
                # thus indptr = [0, 1, 2, 4]
                # see https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csc_matrix.html
                # for more details
                # this is useful for efficient column slicing and matrix-vector products
                # e.g., A[:, 2] can be accessed directly using indptr[2] to indptr[3]
                # without scanning the entire data array
                # this is different from CSR format which is row-based
                # see https://en.wikipedia.org/wiki/Sparse_matrix#Compressed_sparse_column_(CSC)
                # for more info
                # in summary, indptr helps locate the start and end of each column in the data/indices arrays
                # enabling efficient column operations
                # this is particularly useful in applications like BM25 where column operations are common
                # e.g., computing term frequencies across documents
                # thus understanding indptr is crucial for working with CSC matrices effectively
                # especially in information retrieval contexts
                # indptr[j] to indptr[j+1] gives the range of non-zero entries for column j, 0 <= j < n_cols
                # data[indptr[j]:indptr[j+1]] gives the actual non-zero values in column j
                # indices[indptr[j]:indptr[j+1]] gives the corresponding row indices for those values
)

(array([4, 5, 1, 6]),
 array([1, 2, 0, 2], dtype=int32),
 array([0, 1, 2, 4], dtype=int32))

In [4]:
bool((A.data == B.data).all() and (A.indices == B.indices).all() and (A.indptr == B.indptr).all())

True

In [5]:
# Example documents
documents = [
    {"title": "Cat Facts", "text": "Cats are curious animals."},
    {"title": "Dog Facts", "text": "Dogs are loyal and friendly."},
    {"title": "Bird Facts", "text": "Birds can fly and sing."}
]

n_docs = len(documents)

In [6]:
# Build vocabulary and trie
all_tokens = set()
for doc in documents:
    all_tokens.update(doc["text"].lower().split())


In [7]:

trie = marisa_trie.Trie(sorted(all_tokens))
n_vocab = len(trie)

token2id = {token: idx for idx, token in enumerate(trie)}

shape = (n_docs, n_vocab)

In [8]:
shape, trie.items(), token2id

((3, 12),
 [('and', 8),
  ('animals.', 9),
  ('are', 4),
  ('can', 10),
  ('cats', 11),
  ('curious', 5),
  ('fly', 6),
  ('friendly.', 7),
  ('birds', 0),
  ('dogs', 1),
  ('loyal', 2),
  ('sing.', 3)],
 {'and': 0,
  'animals.': 1,
  'are': 2,
  'can': 3,
  'cats': 4,
  'curious': 5,
  'fly': 6,
  'friendly.': 7,
  'birds': 8,
  'dogs': 9,
  'loyal': 10,
  'sing.': 11})

In [9]:
def stream_docs(columnar_texts, token2id ):
    for text in columnar_texts:
        tokens = text.lower().split()  # TODO: improve tokenization
        token_ids = np.array([token2id[token] for token in tokens if token in token2id], dtype=np.int32)
        yield token_ids

In [10]:
# df & nnz_total
df = np.zeros(n_vocab, dtype=np.int32)
doc_len = np.zeros(n_docs, dtype=np.int32)
nnz_total = 0  # total count of unique term occurrences in a document

field_name = "text"

columnar_posting = [doc[field_name] for doc in documents]

for d, terms in enumerate(stream_docs(columnar_posting, trie)):  # terms: np.array of vocab ids (duplicates allowed)
    doc_len[d] = len(terms)
    uniq = np.unique(terms)
    df[uniq] += 1
    nnz_total += uniq.size


In [11]:
nnz_total

14

In [12]:
# idf
idf = np.log((n_docs - df + 0.5) / (df + 0.5))
idf = np.maximum(idf, 0)
idf

array([0.51082562, 0.51082562, 0.51082562, 0.51082562, 0.        ,
       0.51082562, 0.51082562, 0.51082562, 0.        , 0.51082562,
       0.51082562, 0.51082562])

In [13]:
# indptr
indptr = np.empty(n_vocab + 1, dtype=np.int32)
indptr[0] = 0
np.cumsum(df, out=indptr[1:])
indptr

array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32)

In [14]:
indices = np.memmap("indices.bin", dtype=np.int32,   mode="w+", shape=(nnz_total,))
data    = np.memmap("data.bin",    dtype=np.float32, mode="w+", shape=(nnz_total,))
offset  = indptr.copy()

# BM25 준비
k1, b = 1.2, 0.75
avgdl = float(doc_len.mean())
# idf   = ...  # 길이 n_vocab, float32: idf[t] = log((N - df[t] + 0.5)/(df[t] + 0.5) + 1), N=n_docs

columnar_posting = [doc[field_name] for doc in documents]

print(offset.shape)
print(f"offset samples: {offset[:10]}")

for d, terms in enumerate(stream_docs(columnar_posting, trie)):
# for d, terms in enumerate(stream_docs(columnar_posting, token2id)):
    uniq, counts = np.unique(terms, return_counts=True)
    dl = float(doc_len[d])
    tf = counts.astype(np.float32)
    bm25 = idf[uniq] * (tf*(k1+1.0)) / (tf + k1*(1.0 - b + b*dl/avgdl))

    # start = offset[uniq]
    # end   = start + counts.size
    # indices[start:end] = [d] * counts.size
    # data[start:end]    = bm25
    # offset[uniq]      += counts.size

    # 각 단어 컬럼에 바로 써넣음 → indices=문서ID, data=점수
    for v, s in zip(uniq.astype(np.int32), bm25):
        pos = offset[v]
        indices[pos] = d
        data[pos]    = s
        offset[v]   += 1

from scipy.sparse import csc_matrix
X = csc_matrix((data, indices, indptr), shape=(n_docs, n_vocab))


(13,)
offset samples: [ 0  1  2  3  4  6  7  8  9 11]


In [15]:
X.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.5425321 , 0.        , 0.        , 0.        , 0.5425321 ,
        0.        , 0.5425321 ],
       [0.        , 0.49632272, 0.49632272, 0.        , 0.        ,
        0.        , 0.        , 0.49632272, 0.        , 0.        ,
        0.        , 0.        ],
       [0.49632272, 0.        , 0.        , 0.49632272, 0.        ,
        0.        , 0.49632272, 0.        , 0.        , 0.        ,
        0.49632272, 0.        ]], dtype=float32)

In [16]:
data, indices, indptr

(memmap([0.49632272, 0.49632272, 0.49632272, 0.49632272, 0.        ,
         0.        , 0.5425321 , 0.49632272, 0.49632272, 0.        ,
         0.        , 0.5425321 , 0.49632272, 0.5425321 ], dtype=float32),
 memmap([2, 1, 1, 2, 0, 1, 0, 2, 1, 1, 2, 0, 2, 0], dtype=int32),
 array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32))

In [17]:
len(indices), len(data), len(indptr)

(14, 14, 13)

In [18]:
X.shape

(3, 12)

In [29]:
indptr.shape, indptr

((13,),
 array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14], dtype=int32))

In [28]:
data.shape, indices.shape, nnz_total

((14,), (14,), 14)

In [19]:
token2id

{'and': 0,
 'animals.': 1,
 'are': 2,
 'can': 3,
 'cats': 4,
 'curious': 5,
 'fly': 6,
 'friendly.': 7,
 'birds': 8,
 'dogs': 9,
 'loyal': 10,
 'sing.': 11}

In [20]:
trie.items()

[('and', 8),
 ('animals.', 9),
 ('are', 4),
 ('can', 10),
 ('cats', 11),
 ('curious', 5),
 ('fly', 6),
 ('friendly.', 7),
 ('birds', 0),
 ('dogs', 1),
 ('loyal', 2),
 ('sing.', 3)]

## scoring term-query scores of document '0'

In [21]:
X[0, trie["curious"]], X[0, trie["cats"]]

(np.float32(0.5425321), np.float32(0.5425321))